<a href="https://colab.research.google.com/github/shirinR/e-commerce-ai-assistant/blob/main/Multi_Agent_AI_Shopping_Copilot_with_Safety_Guardrails_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Architecture (Simple but Powerful)
User
│
▼
Supervisor Agent
│
├── Intent Agent
├── Product Search Agent
├── Review Analyzer Agent
├── Comparison Agent
│
▼
Governance Layer
├── Safety Agent
├── Compliance Agent
└── Evaluation Agent


Tech Stack (Simple)

Backend:
*   Python
*   FastAPI

Agent orchestration
*  LangGraph  or CrewAI

Vector search
*  FAISS

LLM
*  GPT or open-source

Dataset
* Kaggle product dataset


This project demonstrates:
* LLM agents
* RAG pipelines
* multi-agent orchestration
* AI governance
* evaluation

One-Week Build Plan

Day 1
* Project setup + product dataset.

Day 2
* Vector search (RAG).

Day 3
* Intent + search agents.

Day 4
* Review analyzer + comparison agent.

Day 5
* Safety + compliance agents.

Day 6
* Evaluation metrics.

Day 7
* Simple UI demo.

In [1]:
!pip install -U -q langchain langchain-openai langchain-text-splitters langchain-community bs4

Step 1 — Initialize ChatOpenAI

In [2]:
import os
from langchain_openai import ChatOpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

Step 2 — Intent Agent:
The intent agent converts the user query into structured fields.

In [3]:
from langchain_core.prompts import PromptTemplate

intent_prompt = PromptTemplate.from_template(
    """Extract shopping intent.
    JSON keys: product_type, max_price, key_features
    Query: {query}"""
)

Agent Execution:

In [4]:
def intent_agent(query):
  prompt = intent_prompt.format(query=query)
  response = llm.invoke(prompt)
  return response.content

In [5]:
query = "I'm looking for a smartphone under $500 with a good camera."
response_content = intent_agent(query)
print(response_content)

```json
{
    "product_type": "smartphone",
    "max_price": 500,
    "key_features": ["good camera"]
}
```


read the csv files

In [6]:
!pip install faiss-cpu
import pandas as pd
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')


# List csv files, using the provided 'ecommerce_transactions.csv'.
csv_file = ["/content/sample_data/IKEA_product_catalog.csv"]

# Collect all documents
all_docs = []

for file in csv_file:
  df = pd.read_csv(file)
  # Limit the DataFrame to the first 3000 rows to prevent timeout during vector store creation
  df = df.head(3000)
  print(f"CSV file '{file}' read successfully, processing first {len(df)} rows.") # Added print statement
  for index, row in df.iterrows():
        # Dynamically construct the text string from all available columns
        text_parts = []
        for col in df.columns:
            value = row[col]
            if pd.notna(value): # Check for non-null values
                text_parts.append(f"{col}: {value}")
        text = ", ".join(text_parts)

        if text: # Only add document if text is not empty
            metadata = {"source_file": file, "row_index": index}
            all_docs.append(Document(page_content=text, metadata=metadata))
        else:
            print(f"Warning: Skipping row {index} in {file} due to empty content.")

print(f"Collected {len(all_docs)} documents.") # Added print statement

# Create embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print("OpenAIEmbeddings initialized.") # Added print statement


# Chunking data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200  # chunk overlap (characters)
)
all_splits = text_splitter.split_documents(all_docs)
print(f"Documents chunked into {len(all_splits)} splits.") # Added print statement

# Build FAISS vector store
print("Starting to build FAISS vector store...") # Added print statement
vector_store = FAISS.from_documents(all_splits, embeddings)

print(f"Vector store contains {len(all_splits)} documents after chunking.")

# Save the FAISS index to disk
vector_store.save_local("faiss_index_ikea")
print("FAISS vector store saved to 'faiss_index_ikea'.")

CSV file '/content/sample_data/IKEA_product_catalog.csv' read successfully, processing first 3000 rows.
Collected 3000 documents.
OpenAIEmbeddings initialized.
Documents chunked into 3000 splits.
Starting to build FAISS vector store...
Vector store contains 3000 documents after chunking.
FAISS vector store saved to 'faiss_index_ikea'.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 3 — Product Search Agent:
The product search agent will retrieve relevant product information from the vector store based on the user's query.

In [23]:
import os
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

# Ensure OpenAI API key is set and LLM is initialized
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') # Already set in a previous cell

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Create a retriever from the FAISS vector store, removed country filter for product_compliance_filter to handle
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Define structured output for product search
class Product(BaseModel):
    product_id: str = Field(description="The unique identifier of the product.")
    product_name: str = Field(description="The name of the product.")
    description: str = Field(description="A brief description of the product.")
    price: Optional[float] = Field(description="The price of the product, if available.")

class ProductList(BaseModel):
    products: List[Product] = Field(description="A list of relevant products.")

# Define a prompt template for the product search agent to return structured JSON
product_search_prompt_json = PromptTemplate.from_template(
    """You are a product search assistant. Based on the following context, identify relevant products.
    Filter out any products that do not meet the user's specific query.
    Provide a JSON list of the top 3-5 most relevant products including their product_id, product_name, description, and price.
    If no relevant products are found, return an empty list.
    Context: {context}
    Query: {query}
    Output JSON (format as {schema}):"""
).partial(schema=ProductList.schema_json())

# Define a compliant retriever that filters documents before passing to LLM
def compliant_retriever_func(query):
    # Retrieve raw documents
    raw_docs = retriever.invoke(query)
    # Apply product compliance filter
    filtered_docs = product_compliance_filter(raw_docs)
    return filtered_docs

# Chain the compliant retriever, prompt, and LLM to create the compliant product search agent
product_search_agent_compliant = (
    {"context": compliant_retriever_func, "query": RunnablePassthrough()} # Retrieves & filters context based on query
    | product_search_prompt_json  # Passes context and query to the prompt for JSON output
    | llm  # Invokes the LLM
    | JsonOutputParser(pydantic_object=ProductList) # Parses the output to structured Pydantic object
)

print("Compliant Product Search Agent with structured output initialized.")

# The original product_search_agent is now superseded by product_search_agent_compliant
# and will be used within the orchestration function.

Compliant Product Search Agent with structured output initialized.


/tmp/ipykernel_17298/765456778.py:40: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  ).partial(schema=ProductList.schema_json())


#### Agent Execution:

In [24]:
def product_search_agent_invoke(query):
  response = product_search_agent_compliant.invoke(query)
  return response

# Test the product search agent
test_query = "I need a comfortable sofa for a small living room under $600."
search_results = product_search_agent_invoke(test_query)
print(f"Query: {test_query}\nSearch Results: {search_results}")

NameError: name 'product_compliance_filter' is not defined

### Step 4 — Review Analyzer Agent:

The review analyze agent is a hybrid of retrieval + aggregation + reasoning agent. Its job is not just to find reviews, but to synthesize sentiment, detect patterns, and extract actionable insights.


#### Given a product (or query), the agent should:

* Retrieve relevant reviews
* Analyze sentiment (positive / negative / neutral)
* Extract key themes (e.g., “durability”, “easy assembly”)
* Summarize insights in a structured way



In [ ]:
review_file = ["/content/sample_data/IKEA_product_catalog.csv"]
df_reviews = pd.read_csv(review_file[0]).head(3000) # Fixed: Access the string path from the list
review_docs = []

for row in df_reviews.itertuples(index=False):
    # Fixed: Use 'product_description' for review text and 'product_rating' for rating
    text = f"Review: {str(row.product_description)}"

    # Safely convert product_rating to numeric, coercing errors to NaN
    # Then convert to float if not NaN, otherwise None
    numeric_rating = pd.to_numeric(row.product_rating, errors='coerce')
    rating_value = float(numeric_rating) if not pd.isna(numeric_rating) else None

    metadata = {
        "product_id": str(row.product_id),
        "rating": rating_value
    }

    review_docs.append(Document(page_content=text, metadata=metadata))

Chunking + Embedding + FAISS

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

review_splits = text_splitter.split_documents(review_docs)
review_vector_store = FAISS.from_documents(review_splits, embeddings)
print(f"Review index size: {len(review_splits)}")

Create Retriver(with filtering support)

In [ ]:
# Re-initializing the retriever to ensure it exists in the kernel
review_retriever = review_vector_store.as_retriever(
    search_kwargs={"k": 20}
)
print("Review retriever initialized.")

LLM prompt for Analysis

In [ ]:
analysis_prompt = PromptTemplate.from_template(
    """
You are an expert product review analyst.

Given the following customer reviews, analyze them carefully.

Average Rating: {avg_rating}

Tasks:
1. Classify overall sentiment (positive, neutral, negative)
2. Identify top 3 pros
3. Identify top 3 cons
4. Identify common themes (quality, price, durability, size, usability, etc.)
5. Provide a final summary

Reviews:
{reviews}

Output format:

Overall Sentiment:
...

Top Pros:
- ...
- ...

Top Cons:
- ...
- ...

Key Themes:
- ...

Summary:
...
"""
)

# Build LLM Chain
# Reusing the 'llm' object already initialized as ChatOpenAI
# llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0) # This line is not needed as llm is already defined

review_chain = analysis_prompt | llm

Review Analyzer Agent Function

In [ ]:
def review_analyzer(product_id, query="overall customer feedback"):
    # Step 1: Retrieve reviews using the invoke method
    # Note: We retrieve results and then filter by product_id in metadata
    docs = review_retriever.invoke(query)

    # Filter documents to match the specific product_id
    filtered_docs = [
        doc for doc in docs
        if str(doc.metadata.get("product_id")) == str(product_id)
    ]

    if not filtered_docs:
        return f"No reviews found for product_id: {product_id}."

    # Step 2: Combine text
    reviews_text = "\n".join([doc.page_content for doc in filtered_docs])

    # Step 3: Compute avg rating
    ratings = [
        doc.metadata.get("rating")
        for doc in filtered_docs
        if doc.metadata.get("rating") is not None
    ]

    avg_rating = round(sum(ratings) / len(ratings), 2) if ratings else "N/A"

    # Step 4: Invoke the analysis chain
    response = review_chain.invoke({
        "reviews": reviews_text,
        "avg_rating": avg_rating
    })

    return response.content

In [ ]:
# Let's test with a product ID that exists in your data
# Based on your kernel state, '10018194' is a valid ID
product_id = "12345"

# Ensure the function and retriever are defined before calling
try:
    output = review_analyzer(product_id)
    print(output)
except NameError as e:
    print(f"Error: {e}. Please ensure you have run the cells defining 'review_retriever' and 'review_analyzer'.")

Compliance Agent:

Core Responsibilities

The compliance agent answers questions like:

“Is this product allowed to be shown?”
“Does this response violate policy?”
“Are we recommending restricted or unsafe items?”
“Is pricing / discount compliant with rules?”

Product Compliance Filter (Code)

In [ ]:
def product_compliance_filter(docs):
  compliant_docs = []

  for doc in docs:
    metadata = doc.metadata

    # Rule 1: Must be sellable
    if metadata.get("online_sellable") != True:
      continue

    # Rule 2: Allowed country
    if metadata.get("country") not in ["US"]:
      continue

    # Rule 3: Restricted categories
    if metadata.get("main_category") in ["restricted","weapons"]:
      continue

    compliant_docs.append(doc)

  return compliant_docs

Response Compliance Filter (LLM-Based):
Prompt for validation

In [17]:
from langchain_core.prompts import PromptTemplate

compliance_prompt = PromptTemplate.from_template(
    template="""
You are a compliance officer.

Check if the following response violates any rules:
- No unsafe or harmful advice
- No false claims
- No restricted product recommendations

If compliant → return: APPROVED
If not → return: REJECTED + reason

Response:
{response}
"""
)

In [18]:
compliance_chain = compliance_prompt | llm

In [19]:
# validation function
def validate_response(response_text):
  result = compliance_chain.invoke({"response": response_text})

  output = result.content.strip()

  if "REJECTED" in output:
      return False, output
  return True, output

Full compliance Agent

In [20]:
def compliance_agent(product_docs, generated_response):

    # Step 1: Filter products
    filtered_docs = product_compliance_filter(product_docs)

    if not filtered_docs:
        return None, "No compliant products available."

    # Step 2: Validate response
    is_valid, reason = validate_response(generated_response)

    if not is_valid:
        return None, f"Response blocked: {reason}"

    return filtered_docs, generated_response

### Step 7 — Evaluation Agent:

The Evaluation Agent will be responsible for assessing the performance and reliability of the other agents in the system. It will analyze outputs, identify potential issues, and provide metrics to ensure the system is meeting its objectives.

#### Intent Agent Evaluation:

Let's start by evaluating the `intent_agent`. This function will check if the `intent_agent` successfully extracts the `product_type`, `max_price`, and `key_features` from a user query.

In [ ]:
def evaluate_intent_agent(user_query):
    print(f"Evaluating intent for query: '{user_query}'")
    intent_json_str_raw = intent_agent(user_query)
    intent = {}
    try:
        # Extract JSON string from markdown code block
        match = re.search(r"```json\n([\s\S]*?)\n```", intent_json_str_raw)
        if match:
            intent_json_str = match.group(1)
        else:
            intent_json_str = intent_json_str_raw # Fallback if not wrapped
        intent = json.loads(intent_json_str)
    except json.JSONDecodeError as e:
        print(f"Error parsing intent JSON: {e}\nRaw output: {intent_json_str_raw}")
        return {"success": False, "reason": "JSON parsing error"}

    product_type_present = "product_type" in intent and intent["product_type"] not in ["", None]
    max_price_present = "max_price" in intent and intent["max_price"] is not None
    key_features_present = "key_features" in intent and isinstance(intent["key_features"], list) and len(intent["key_features"]) > 0

    if product_type_present and max_price_present and key_features_present:
        return {"success": True, "details": intent}
    else:
        reasons = []
        if not product_type_present: reasons.append("product_type missing or empty")
        if not max_price_present: reasons.append("max_price missing")
        if not key_features_present: reasons.append("key_features missing or empty")
        return {"success": False, "reason": ", ".join(reasons), "details": intent}

# Let's test this evaluation function
query_good = "I need a comfortable sofa under $1000 with washable covers."
query_bad = "I want something to sit on."

print("\n--- Evaluation for Good Query ---")
result_good = evaluate_intent_agent(query_good)
print(result_good)

print("\n--- Evaluation for Bad Query ---")
result_bad = evaluate_intent_agent(query_bad)
print(result_bad)

#### Product Search Agent Evaluation:

Now, let's evaluate the `product_search_agent_compliant`. This function will check if the returned products align with the user's query and the extracted intent.

In [ ]:
def evaluate_product_search_agent(user_query, expected_intent):
    print(f"\nEvaluating product search for query: '{user_query}'")
    print(f"Expected intent: {expected_intent}")

    search_results = product_search_agent_invoke(user_query)
    products = search_results.products if hasattr(search_results, 'products') else []

    if not products:
        return {"success": False, "reason": "No products returned", "details": search_results}

    max_price = expected_intent.get('max_price')
    key_features = [f.lower() for f in expected_intent.get('key_features', [])]
    product_type = expected_intent.get('product_type', '').lower()

    all_products_match = True
    evaluation_details = []

    for product in products:
        product_match = True
        product_info = {
            "product_id": product.product_id,
            "product_name": product.product_name,
            "price": product.price,
            "description": product.description
        }

        # Check price compliance
        if max_price is not None and product.price is not None and product.price > max_price:
            product_match = False
            product_info["price_check"] = f"FAIL: {product.price} > {max_price}"
        else:
            product_info["price_check"] = "PASS"

        # Check key features presence in description
        found_features = []
        missing_features = []
        if key_features:
            for feature in key_features:
                if feature in product.description.lower():
                    found_features.append(feature)
                else:
                    missing_features.append(feature)
            if missing_features:
                product_match = False
                product_info["features_check"] = f"FAIL: Missing features: {', '.join(missing_features)}"
            else:
                product_info["features_check"] = "PASS"
        else:
            product_info["features_check"] = "N/A (no key features expected)"

        # Basic product type check (can be expanded if product_type is more explicitly in description/metadata)
        if product_type and product_type not in product.product_name.lower() and product_type not in product.description.lower():
            # This check is less strict as product_type might be broader than individual product name/description
            pass # For now, we'll allow it if other checks pass, or make this stricter later

        if not product_match:
            all_products_match = False

        evaluation_details.append(product_info)

    if all_products_match:
        return {"success": True, "details": evaluation_details, "summary": "All returned products seem to match criteria."}
    else:
        return {"success": False, "reason": "Some products did not meet criteria.", "details": evaluation_details}

# --- Test the Product Search Agent Evaluation Function ---

# Example 1: Good query with clear expectations
query_ps_good = "I need a comfortable sofa for a small living room under $600."
expected_intent_ps_good = {
    "product_type": "sofa",
    "max_price": 600,
    "key_features": ["comfortable", "small living room"]
}

result_ps_good = evaluate_product_search_agent(query_ps_good, expected_intent_ps_good)
print("\n--- Evaluation for Good Product Search Query ---")
print(result_ps_good)

# Example 2: Query with higher price expectation (might fail if products are too expensive)
query_ps_mid = "I'm looking for a modern dining table under $1200."
expected_intent_ps_mid = {
    "product_type": "dining table",
    "max_price": 1200,
    "key_features": ["modern"]
}

result_ps_mid = evaluate_product_search_agent(query_ps_mid, expected_intent_ps_mid)
print("\n--- Evaluation for Mid-Range Product Search Query ---")
print(result_ps_mid)

# Example 3: Query with no matching products expected (should return success=False, reason=No products returned)
query_ps_bad = "I need a flying car for under $100."
expected_intent_ps_bad = {
    "product_type": "flying car",
    "max_price": 100,
    "key_features": ["fast"]
}

result_ps_bad = evaluate_product_search_agent(query_ps_bad, expected_intent_ps_bad)
print("\n--- Evaluation for Bad Product Search Query ---")
print(result_ps_bad)

### Orchestration Agent:
This agent acts as the central coordinator, orchestrating the flow of information between the Intent Agent, Product Search Agent, Review Analyzer Agent, and Compliance Agent. It takes a user query, processes it through each specialized agent, and compiles a comprehensive and compliant response.

In [21]:
import re
def orchestration_agent(user_query):
    # 1. Intent Agent: Extract structured intent from the user query
    print(f"Orchestration: Processing intent for query: '{user_query}'")
    intent_json_str_raw = intent_agent(user_query)
    try:
        # Extract JSON string from markdown code block
        match = re.search(r"```json\n([\s\S]*?)\n```", intent_json_str_raw)
        if match:
            intent_json_str = match.group(1)
        else:
            intent_json_str = intent_json_str_raw # Fallback if not wrapped

        intent = json.loads(intent_json_str)
    except json.JSONDecodeError:
        return f"Orchestration: Error parsing intent: {intent_json_str_raw}"

    product_type = intent.get("product_type", "")
    max_price = intent.get("max_price")
    key_features = intent.get("key_features", [])

    # Construct a detailed query for the product search agent
    search_query = f"I am looking for a {product_type}."
    if max_price:
        search_query += f" The price should be under ${max_price}.";
    if key_features:
        search_query += f" It should have the following features: {', '.join(key_features)}."

    # 2. Product Search Agent: Retrieve compliant products
    print(f"Orchestration: Searching for products with query: '{search_query}'")
    try:
        product_list_output = product_search_agent_compliant.invoke(search_query)
        products = []
        if isinstance(product_list_output, ProductList): # Check if it's the Pydantic object
            products = product_list_output.products
        elif isinstance(product_list_output, dict) and 'products' in product_list_output: # Check if it's a dictionary with 'products' key
            products = product_list_output['products']
        elif isinstance(product_list_output, list): # If LLM returned a direct list (e.g., an empty list [])
            products = product_list_output
        # else: products remains an empty list, and the 'if not products' block will handle it
    except Exception as e:
        return f"Orchestration: Error during product search: {e}"

    if not products:
        final_response = f"No products found matching your criteria: {user_query}"
        # Even if no products, validate the response for compliance (e.g., safe wording)
        is_valid, reason = validate_response(final_response)
        if not is_valid:
            return f"Response blocked by compliance: {reason}"
        return final_response

    response_parts = ["Here are some products that match your request:"]
    all_found_product_ids = []

    for product in products:
        # Assuming 'product' itself is a dictionary from the LLM output if the Pydantic parsing failed
        product_id = product.get('product_id', 'N/A')
        product_name = product.get('product_name', 'Unknown Product')
        description = product.get('description', 'No description available.')
        price = product.get('price')

        all_found_product_ids.append(product_id)
        product_details = f"\n- **{product_name}** (ID: {product_id})\n  Description: {description}\n"
        if price is not None:
            product_details += f"  Price: ${price:.2f}\n"

        # 3. Review Analyzer Agent: Get review insights for each product
        print(f"Orchestration: Analyzing reviews for product ID: {product_id}")
        review_summary = review_analyzer(product_id, query=f"Reviews for {product_name}")
        product_details += f"  Customer Feedback: {review_summary.strip().replace('Overall Sentiment:', 'Sentiment:').replace('Top Pros:', 'Pros:').replace('Top Cons:', 'Cons:').replace('Key Themes:', 'Themes:').replace('Summary:', 'Summary:')}"

        response_parts.append(product_details)

    final_response = "\n".join(response_parts)

    # 4. Compliance Agent (Response Validation only):
    # Product compliance was handled by compliant_retriever_func within product_search_agent_compliant.
    # Now, validate the generated natural language response.
    print("Orchestration: Validating final response for compliance...")
    is_valid, reason = validate_response(final_response)

    if not is_valid:
        return f"Response blocked by compliance: {reason}"
    else:
        print("Orchestration: Response is compliant.")
        return final_response


import json # Make sure json is imported for parsing intent

In [22]:
test_orchestration_query = "I'm looking for a sofa between $1500 and $1800."
print(f"Testing orchestration agent with query: {test_orchestration_query}")
orchestration_result = orchestration_agent(test_orchestration_query)
print("\n--- Orchestration Agent Result ---")
print(orchestration_result)

Testing orchestration agent with query: I'm looking for a sofa between $1500 and $1800.
Orchestration: Processing intent for query: 'I'm looking for a sofa between $1500 and $1800.'
Orchestration: Searching for products with query: 'I am looking for a sofa. The price should be under $1800. It should have the following features: min_price.'

--- Orchestration Agent Result ---
Orchestration: Error during product search: name 'product_compliance_filter' is not defined
